# Loading the Dataset

In [ ]:
# Authenticate with your GCP account
from google.colab import auth
auth.authenticate_user()

!pip install -q gcsfs

import pandas as pd
df = pd.read_parquet("gs://ba865-t9-mimiciv/parquet/cohort_model_input")

print("Shape:", df.shape)
df.head(3)

Shape: (272842, 13)


,hadm_id,readmitted_30d,anchor_age,gender,los_days,prior_admissions,icd10_chapter,admission_type,insurance,creatinine,hemoglobin,heart_rate_discharge,discharge_text
0,20000502,1,49,M,4.71,0,8,EW EMER.,Private,0.8,12.8,82.0,\nName: ___ Unit No: ___\...
1,20002493,0,70,F,11.90,0,C,OBSERVATION ADMIT,Medicare,0.8,7.9,82.0,\nName: ___ Unit No: ___...
2,20003332,0,57,M,5.92,0,E,OBSERVATION ADMIT,Private,0.6,14.0,82.0,\nName: ___ Unit No: ___\n...


# EDA and Preprocessing

In [ ]:
# check dataset shape and structure
print("Shape:", df.shape)
# inspect data types of each column
print("\nColumn types:")
print(df.dtypes)

Shape: (272842, 13)

Column types:
hadm_id                   int32
readmitted_30d            int32
anchor_age                int32
gender                   object
los_days                float64
prior_admissions          int32
icd10_chapter            object
admission_type           object
insurance                object
creatinine              float64
hemoglobin              float64
heart_rate_discharge    float64
discharge_text           object
dtype: object


In [ ]:
# calculate missing values and percentage
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df)) * 100

pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct})

,missing_count,missing_pct
insurance,3048,1.11713
readmitted_30d,0,0.00000
anchor_age,0,0.00000
gender,0,0.00000
hadm_id,0,0.00000
los_days,0,0.00000
prior_admissions,0,0.00000
icd10_chapter,0,0.00000
admission_type,0,0.00000
creatinine,0,0.00000


In [ ]:
# check class distribution of readmission variable
df['readmitted_30d'].value_counts(), df['readmitted_30d'].value_counts(normalize=True)

(readmitted_30d
 0    211694
 1     61148
 Name: count, dtype: int64,
 readmitted_30d
 0    0.775885
 1    0.224115
 Name: proportion, dtype: float64)

In [ ]:
# compute mean values grouped by target
num_cols = ['anchor_age', 'los_days', 'prior_admissions',
            'creatinine', 'hemoglobin', 'heart_rate_discharge']

df.groupby('readmitted_30d')[num_cols].mean()

,anchor_age,los_days,prior_admissions,creatinine,hemoglobin,heart_rate_discharge
readmitted_30d,,,,,,
0,59.919067,5.649381,1.936328,1.138807,10.971250,82.106505
1,59.550991,7.013293,4.037139,1.312977,10.369446,82.357022


In [ ]:
# view value counts for categorical variables
cat_cols = ['gender', 'icd10_chapter', 'admission_type', 'insurance']

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(10))


--- gender ---
gender
F    137377
M    135465
Name: count, dtype: int64

--- icd10_chapter ---
icd10_chapter
4    40658
5    33127
I    21515
2    19340
7    18161
K    14086
9    13117
8    10623
1     8698
0     8438
Name: count, dtype: int64

--- admission_type ---
admission_type
EW EMER.                       135570
OBSERVATION ADMIT               49970
SURGICAL SAME DAY ADMISSION     30435
URGENT                          27587
DIRECT EMER.                    18045
ELECTIVE                         9834
EU OBSERVATION                    901
DIRECT OBSERVATION                455
AMBULATORY OBSERVATION             45
Name: count, dtype: int64

--- insurance ---
insurance
Medicare     136814
Private       82604
Medicaid      43137
Other          7193
No charge        46
Name: count, dtype: int64


In [ ]:
# compute length of discharge notes
df['text_length'] = df['discharge_text'].astype(str).apply(len)
df['text_length'].describe()

,text_length
count,272842.000000
mean,10883.954992
std,4526.047013
min,353.000000
25%,7741.000000
50%,10193.000000
75%,13224.000000
max,60381.000000


In [ ]:
# view sample discharge notes
for i in range(3):
    print(f"\n--- Sample {i} ---\n")
    print(df['discharge_text'].iloc[i][:800])


--- Sample 0 ---

 
Name:  ___                  Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: SURGERY
 
Allergies: 
Patient recorded as having No Known Allergies to Drugs
 
Attending: ___.
 
Chief Complaint:
s/p 14 foot Fall
 
Major Surgical or Invasive Procedure:
___ ORIF right distal tibia
 
History of Present Illness:
___ yo male s/p fall from approx 15 ft off of ladder, no reported 
LOC. transported to ___ for further care.
 
Past Medical History:
Denies
 
Family History:
Noncontributory
 
Physical Exam:
On presentation:
Gen: NAD, in a C-Collar
HEENT:  EOMI, PERRL, trachea is midline, no LAD
RESP: CTA-B
CV: RRR, nl S1 and S2, no MRG
ABD: soft, NT/ND, +BS, normal rectal tone with no gross blood.
EXT: Pelvis is stable, ___ st

--- Sample 1 ---

 
Name:  ___                   Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: SUR

In [ ]:
# drop hadm_id since it is a unique identifier
df = df.drop(columns=['hadm_id'])

In [ ]:
# fill missing insurance values
df['insurance'] = df['insurance'].fillna('Unknown')

In [ ]:
# clean discharge text (light cleaning only)
# remove placeholders
df['discharge_text'] = df['discharge_text'].str.replace('___', '', regex=False)

# remove newlines
df['discharge_text'] = df['discharge_text'].str.replace('\n', ' ', regex=False)

# fix multiple spaces (use raw string)
df['discharge_text'] = df['discharge_text'].str.replace(r'\s+', ' ', regex=True)

# Preparation for Modelling

In [ ]:
# one-hot encode categorical columns
df = pd.get_dummies(df, columns=['gender', 'icd10_chapter', 'admission_type', 'insurance'], drop_first=True)

In [ ]:
# take smaller sample first
df_sample = df.sample(n=50000, random_state=42)

In [ ]:
# define target
y = df_sample['readmitted_30d']

# drop target from features
X = df_sample.drop(columns=['readmitted_30d'])

In [ ]:
# separate text column
X_text = X['discharge_text']

# drop text from structured features
X_struct = X.drop(columns=['discharge_text'])

In [ ]:
# split data into train and test
from sklearn.model_selection import train_test_split

X_struct_train, X_struct_test, X_text_train, X_text_test, y_train, y_test = train_test_split(
    X_struct, X_text, y, test_size=0.2, random_state=42, stratify=y)

KeyboardInterrupt: 

In [ ]:
# check split shapes
print("X_struct_train:", X_struct_train.shape)
print("X_struct_test:", X_struct_test.shape)
print("X_text_train:", X_text_train.shape)
print("X_text_test:", X_text_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
# scale structured features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_struct_train = scaler.fit_transform(X_struct_train)
X_struct_test = scaler.transform(X_struct_test)

In [ ]:
# convert text to list format
X_text_train = X_text_train.tolist()
X_text_test = X_text_test.tolist()

In [ ]:
!pip install sentence-transformers -q

In [ ]:
# load sentence transformer model
from sentence_transformers import SentenceTransformer

model_sbert = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# test SBERT on 3 notes
sample_embeddings = model_sbert.encode(X_text_train[:3], show_progress_bar=True)
sample_embeddings.shape

In [ ]:
# embed training text
X_text_train_emb = model_sbert.encode(X_text_train, show_progress_bar=True)

In [ ]:
# embed test text
X_text_test_emb = model_sbert.encode(X_text_test, show_progress_bar=True)

# Models

## Model 1: Baseline Dense Neural Network (SBERT + Structured)
Combined SBERT text embeddings and structured features into a single input and trained a simple dense neural network. Achieved high accuracy (~77%) but very poor recall (~16%), indicating strong bias toward the majority class.

In [ ]:
# combine structured + text embeddings
import numpy as np

X_train_final = np.hstack([X_struct_train, X_text_train_emb])
X_test_final = np.hstack([X_struct_test, X_text_test_emb])

In [ ]:
# build model
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(X_train_final.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')])

In [ ]:
# compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy'])

In [ ]:
# train model
history = model.fit(
    X_train_final, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32)

In [ ]:
# evaluate model on test set
test_loss, test_acc = model.evaluate(X_test_final, y_test)

print("Test Accuracy:", test_acc)
print("Test Loss:", test_loss)

In [ ]:
# predict again after class-weighted training
y_pred_prob = model.predict(X_test_final)
y_pred = (y_pred_prob > 0.5).astype(int)

# classification report after class weights
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.xticks(epochs)
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, history.history['loss'], label='Train Loss')
plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

## Model 2: Class-Weighted Dense Model
Introduced class weights to address imbalance while keeping the same single-input dense architecture. Improved recall significantly (~48%) at the cost of lower overall accuracy.

In [ ]:
# compute class weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train)

class_weights = {0: class_weights[0], 1: class_weights[1]}
class_weights

In [ ]:
# train model with class weights
history = model.fit(
    X_train_final, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32,
    class_weight=class_weights)

In [ ]:
# predict again after class-weighted training
y_pred_prob = model.predict(X_test_final)
y_pred = (y_pred_prob > 0.5).astype(int)

# classification report after class weights
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

Introducing class weights significantly improved recall for readmitted patients (from 16% to 48%), demonstrating the importance of addressing class imbalance. While overall accuracy decreased, the model became more effective at identifying high-risk patients, which is more aligned with the clinical objective.

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.xticks(epochs)
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, history.history['loss'], label='Train Loss')
plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

## Model 3: Multimodal Neural Network (Functional API)
Built a two-branch model to separately process structured features and SBERT embeddings before combining them. Improved representation learning and increased recall further (~58%) with better use of both data types.

In [ ]:
# build multimodal model using Functional API
from tensorflow.keras import layers, models

# structured input
input_struct = layers.Input(shape=(X_struct_train.shape[1],))
x1 = layers.Dense(64, activation='relu')(input_struct)
x1 = layers.Dropout(0.3)(x1)

# text input (embeddings)
input_text = layers.Input(shape=(X_text_train_emb.shape[1],))
x2 = layers.Dense(128, activation='relu')(input_text)
x2 = layers.Dropout(0.3)(x2)

# combine
combined = layers.concatenate([x1, x2])

# final layers
x = layers.Dense(64, activation='relu')(combined)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=[input_struct, input_text], outputs=output)

In [ ]:
# compile multimodal model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

In [ ]:
# # early stopping callback
# from tensorflow.keras.callbacks import EarlyStopping

# early_stop = EarlyStopping(
#     monitor='val_loss',
#     patience=2,
#     restore_best_weights=True)

In [ ]:
# train multimodal model
history = model.fit(
    [X_struct_train, X_text_train_emb], y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights)

In [ ]:
# predictions
y_pred_prob = model.predict([X_struct_test, X_text_test_emb])
y_pred = (y_pred_prob > 0.5).astype(int)

# classification report
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.xticks(epochs)
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, history.history['loss'], label='Train Loss')
plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

## Model 4: Regularized Multimodal Model
Added dropout and batch normalization to the multimodal architecture to reduce overfitting and stabilize training. Further improved recall (~67%) and made the model more aligned with the goal of identifying high-risk patients.

In [ ]:
# improved multimodal model with better regularization
from tensorflow.keras import layers, models

# structured branch
input_struct = layers.Input(shape=(X_struct_train.shape[1],))
x1 = layers.Dense(64, activation='relu')(input_struct)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Dropout(0.4)(x1)

# text branch
input_text = layers.Input(shape=(X_text_train_emb.shape[1],))
x2 = layers.Dense(128, activation='relu')(input_text)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Dropout(0.4)(x2)

# combine
combined = layers.concatenate([x1, x2])

# final layers
x = layers.Dense(64, activation='relu')(combined)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=[input_struct, input_text], outputs=output)

In [ ]:
# compile improved model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

In [ ]:
# train improved model
history = model.fit(
    [X_struct_train, X_text_train_emb], y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights)

In [ ]:
# predict on test set
y_pred_prob = model.predict([X_struct_test, X_text_test_emb])
y_pred = (y_pred_prob > 0.5).astype(int)

# classification report
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.xticks(epochs)
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, history.history['loss'], label='Train Loss')
plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

## Model 5: Multimodal Model with Checkpointing & LR Scheduling
Incorporated ModelCheckpoint and ReduceLROnPlateau to control training and retain the best-performing model. Achieved the most stable and reliable performance with strong recall (~60%+) and improved generalization.

In [ ]:
# callbacks for better training control
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

checkpoint = ModelCheckpoint(
    "best_sbert_multimodal.keras",
    monitor="val_auc",
    mode="max",
    save_best_only=True,
    verbose=1)

lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    verbose=1)

In [ ]:
# SBERT multimodal model
from tensorflow.keras import layers, models

# structured branch
input_struct = layers.Input(shape=(X_struct_train.shape[1],), name='structured_input')
x1 = layers.Dense(64, activation='relu')(input_struct)
x1 = layers.Dropout(0.3)(x1)

# text embedding branch
input_text = layers.Input(shape=(X_text_train_emb.shape[1],), name='text_input')
x2 = layers.Dense(128, activation='relu')(input_text)
x2 = layers.Dropout(0.3)(x2)

# combine branches
combined = layers.concatenate([x1, x2])

# classifier head
x = layers.Dense(64, activation='relu')(combined)
x = layers.Dropout(0.3)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=[input_struct, input_text], outputs=output)

In [ ]:
# compile model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

In [ ]:
# train model with checkpoint and LR scheduler
history = model.fit(
    [X_struct_train, X_text_train_emb], y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[checkpoint, lr_scheduler])

In [ ]:
# load best model
from tensorflow.keras.models import load_model
model = load_model("best_sbert_multimodal.keras")

In [ ]:
# predictions
y_pred_prob = model.predict([X_struct_test, X_text_test_emb])
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history.history['accuracy']) + 1)

plt.figure(figsize=(12,5))

# Accuracy plot
plt.subplot(1,2,1)
plt.plot(epochs, history.history['accuracy'], label='Train Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.xticks(epochs)
plt.legend()

# Loss plot
plt.subplot(1,2,2)
plt.plot(epochs, history.history['loss'], label='Train Loss')
plt.plot(epochs, history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.xticks(epochs)
plt.legend()

plt.tight_layout()
plt.show()

# **Model 6**

In [ ]:
# split each discharge note into fixed-size text chunks
def split_into_chunks(text, chunk_size=1200, max_chunks=5):
    text = str(text)
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    return chunks[:max_chunks]

In [ ]:
# embed chunks and pad to fixed number of chunks
import numpy as np

def embed_chunked_notes(chunked_texts, max_chunks=5, embedding_dim=384):
    all_embeddings = []

    for chunks in chunked_texts:
        chunk_emb = model_sbert.encode(chunks, show_progress_bar=False)

        # pad if note has fewer than max_chunks
        if len(chunk_emb) < max_chunks:
            padding = np.zeros((max_chunks - len(chunk_emb), embedding_dim))
            chunk_emb = np.vstack([chunk_emb, padding])

        all_embeddings.append(chunk_emb[:max_chunks])

    return np.array(all_embeddings)

In [ ]:
# generate chunk-level embeddings
X_text_train_chunks_emb = embed_chunked_notes(X_text_train_chunks)
X_text_test_chunks_emb = embed_chunked_notes(X_text_test_chunks)

# check shape
print(X_text_train_chunks_emb.shape)